# ad fontes — DPO training (Colab / Kaggle free tier)

Tunes **Qwen2.5-1.5B-Instruct** toward *faithful-and-humble* on the preference
pairs from `app/rlhf/build_pairs.py` (`data/rlhf/pairs.jsonl`), using **QLoRA
4-bit + TRL `DPOTrainer`**, β ≈ 0.1, 1–2 epochs.

**Constraints (brief §2):** free T4 (~15 GB), sessions die → this notebook
**checkpoints to Drive and resumes automatically**. Model ≤ 2B, QLoRA only.

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Upload your `data/rlhf/pairs.jsonl` (from `python -m app.rlhf.build_pairs`) to
   `MyDrive/ad-fontes/pairs.jsonl`.
3. Run every cell top to bottom. Re-running after a disconnect picks up from the
   last checkpoint automatically.

Output: a LoRA adapter at `MyDrive/ad-fontes/qlora-dpo/`. Merge + GGUF conversion
are in `app/rlhf/export_gguf.md`.


## 1. Setup


In [ ]:
# Pinned to what this notebook was written against. Colab already has torch.
%pip install -q "transformers>=4.46,<4.48" "trl==0.13.0" "peft==0.14.0" "datasets>=3.0,<4" "accelerate>=1.0,<2" "bitsandbytes>=0.44,<0.46"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, textwrap
from pathlib import Path

# ------------------------------------------------------------------ config
DRIVE      = Path("/content/drive/MyDrive/ad-fontes")
PAIRS      = DRIVE / "pairs.jsonl"          # upload data/rlhf/pairs.jsonl here
CKPT_DIR   = DRIVE / "dpo-checkpoints"      # resumable checkpoints
ADAPTER_OUT= DRIVE / "qlora-dpo"            # final adapter
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

BETA          = 0.1
NUM_EPOCHS    = 2
LR            = 5e-6
MAX_LEN       = 1024
MAX_PROMPT    = 896
LORA_R        = 16
BATCH         = 2
GRAD_ACCUM    = 8      # effective batch 16
SAVE_STEPS    = 25
SEED          = 0

CKPT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_OUT.mkdir(parents=True, exist_ok=True)
assert PAIRS.exists(), f"upload data/rlhf/pairs.jsonl to {PAIRS}"

import torch, transformers, trl, peft
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| trl", trl.__version__, "| peft", peft.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (set runtime to T4)")


## 2. Data


In [ ]:
from datasets import Dataset

rows = [json.loads(l) for l in PAIRS.read_text().splitlines() if l.strip()]
assert rows and set(rows[0]) >= {"prompt", "chosen", "rejected"}, rows[0].keys()

ds = Dataset.from_list([{"prompt": r["prompt"], "chosen": r["chosen"], "rejected": r["rejected"]}
                        for r in rows]).shuffle(seed=SEED)
split = ds.train_test_split(test_size=min(0.1, 40 / len(ds)), seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

lens = sorted(len(r["prompt"]) + len(r["chosen"]) for r in rows)
print(f"{len(rows)} pairs -> {len(train_ds)} train / {len(eval_ds)} eval")
print(f"prompt+chosen chars: p50 {lens[len(lens)//2]}, p95 {lens[int(len(lens)*0.95)]}")
print("example prompt tail:", train_ds[0]["prompt"][-160:])
print("example chosen    :", train_ds[0]["chosen"][:200])


## 3. Model (QLoRA 4-bit) + LoRA


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
# TRL's DPO collator sets its own padding; leave the tokenizer default.

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, torch_dtype=torch.bfloat16, device_map={"": 0}
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_R * 2, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)


## 4. DPO config + trainer


In [ ]:
from trl import DPOConfig, DPOTrainer

cfg = DPOConfig(
    output_dir=str(CKPT_DIR),
    beta=BETA,
    loss_type="sigmoid",
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    max_length=MAX_LEN,
    max_prompt_length=MAX_PROMPT,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,          # keep Drive small; resume still works
    report_to=[],               # set ["wandb"] + %pip install wandb if you want it
    seed=SEED,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,             # peft_config given -> reference = adapter-disabled base
    args=cfg,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tok,
    peft_config=lora,
)


## 5. Reference-model log-prob sanity check (brief §6)

The DPO loss compares policy vs. reference log-probs of `chosen`/`rejected`. If
the reference numbers are non-finite or positive, training is broken before it
starts (usually a dtype or padding-side bug).


In [ ]:
import torch

trainer.model.eval()
batch = trainer.data_collator([trainer.train_dataset[i] for i in range(min(4, len(train_ds)))])
batch = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in batch.items()}
with torch.no_grad():
    ref = trainer.compute_ref_log_probs(batch)          # (chosen_logps, rejected_logps)
chosen_lp, rejected_lp = ref
print("ref logps  chosen:", chosen_lp.tolist())
print("ref logps rejected:", rejected_lp.tolist())
assert torch.isfinite(chosen_lp).all() and torch.isfinite(rejected_lp).all(), "non-finite!"
assert (chosen_lp < 0).all() and (rejected_lp < 0).all(), "log-probs must be negative!"
print("\n✓ reference log-probs look sane")
trainer.model.train()


## 6. Train (auto-resume)

If the session died, just re-run this cell — it continues from the newest
checkpoint in `CKPT_DIR`. On a T4 expect roughly 20–40 min for ~1k pairs × 2 epochs.


In [ ]:
last_ckpt = None
if any(CKPT_DIR.glob("checkpoint-*")):
    last_ckpt = str(sorted(CKPT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))[-1])
    print("resuming from", last_ckpt)

trainer.train(resume_from_checkpoint=last_ckpt)

trainer.save_model(str(ADAPTER_OUT))
tok.save_pretrained(str(ADAPTER_OUT))
print("adapter saved to", ADAPTER_OUT)


## 7. Training curve + final metrics


In [ ]:
import pandas as pd
hist = pd.DataFrame(trainer.state.log_history)
cols = [c for c in ["loss", "eval_loss", "rewards/accuracies", "rewards/margins"] if c in hist]
print(hist[["step"] + cols].dropna(how="all", subset=cols).tail(15).to_string(index=False))
hist.to_csv(DRIVE / "dpo-log.csv", index=False)

final = hist.dropna(subset=["rewards/margins"]).iloc[-1] if "rewards/margins" in hist else None
if final is not None:
    print(f"\nfinal  rewards/accuracies={final['rewards/accuracies']:.3f}  "
          f"rewards/margins={final['rewards/margins']:.3f}  loss={final['loss']:.3f}")
    print("(accuracies > 0.5 and positive margins => the model prefers 'chosen')")


## 8. Eyeball: base vs. tuned on a few held-out prompts


In [ ]:
import contextlib, textwrap, torch

pol = trainer.model
pol.eval()

def _gen(prompt):
    ids = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT).to(pol.device)
    with torch.no_grad():
        out = pol.generate(**ids, max_new_tokens=220, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

def tuned(prompt):
    return _gen(prompt)

def base(prompt):
    with pol.disable_adapter():          # peft: run the frozen base
        return _gen(prompt)

for r in eval_ds.select(range(min(3, len(eval_ds)))):
    q = r["prompt"].split("Question:")[-1].strip()[:120]
    print("Q:", q)
    print("  BASE :", textwrap.shorten(base(r["prompt"]).replace(chr(10), " "), 280))
    print("  TUNED:", textwrap.shorten(tuned(r["prompt"]).replace(chr(10), " "), 280))
    print()
pol.train()


## 9. Next

Adapter is at `MyDrive/ad-fontes/qlora-dpo/`. Follow
[`app/rlhf/export_gguf.md`](export_gguf.md):

1. merge the adapter into the fp16 base,
2. `convert_hf_to_gguf.py` → `llama-quantize Q4_K_M`,
3. upload to `DEMONKINGKAI/ad-fontes-generator-1.5b-dpo-gguf`.

Then `python -m app.eval.run_eval --stage compare` (Phase 5) scores base vs. tuned.

**Record in `export_gguf.md`:** the versions above, final loss / accuracies / margins,
the reference-log-prob output, wall-clock, and the GGUF size.
